In [ ]:
%pip install --upgrade pip
%pip install torch --index-url https://download.pytorch.org/whl/cpu
%pip install --upgrade transformers

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 9.6 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
MODEL_DIR = Path("smollm2_model")

print("Downloading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Downloading model...")

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

print("Saving model locally...")

tokenizer.save_pretrained(MODEL_DIR)
model.save_pretrained(MODEL_DIR, safe_serialization=True)

print("Model downloaded successfully!")
print("Model location:", MODEL_DIR.resolve())

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

C:\Users\DELL\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM2-360M-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Saving model locally...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model downloaded successfully!
Model location: C:\Users\DELL\Desktop\DS_1.30Batch\smollm2_model


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(4)

MODEL_DIR = "smollm2_model"

print("Loading local model...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

model.to("cpu")
model.eval()

print("Model loaded successfully!")
print("Running on CPU.")

Loading local model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Running on CPU.


In [3]:
SYSTEM_PROMPT = """
You are a helpful AI teaching assistant.
Explain concepts in simple English.
Use examples whenever helpful.
If you are unsure about an answer, say so.
"""

MAX_NEW_TOKENS = 120
MAX_HISTORY_MESSAGES = 6


def generate_response(user_message, history=None):

    if history is None:
        history = []

    # Keep recent conversation messages
    recent_history = history[-MAX_HISTORY_MESSAGES:]

    # Prepare the conversation
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ] + recent_history + [
        {
            "role": "user",
            "content": user_message
        }
    ]

    # Convert conversation into model input
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    # Generate a response
    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Extract only newly generated tokens
    input_length = inputs["input_ids"].shape[1]

    new_tokens = output_ids[0][input_length:]

    # Convert tokens to readable text
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    if not answer:
        answer = "I could not generate a response. Please try again."

    return answer


print("Chatbot function created successfully!")

Chatbot function created successfully!


In [4]:
question = "Explain Generative AI in simple terms."

answer = generate_response(question)

print("You:", question)
print("\nGenAI Chatbot:", answer)

You: Explain Generative AI in simple terms.

GenAI Chatbot: Generative AI is a type of artificial intelligence that creates new data or content from existing ones. It uses algorithms and mathematical formulas to generate something entirely different from what was given as input. This can be anything from creating art, music, or even writing stories, but it can also be used for generating text like sentences or paragraphs.

Think of it like this: imagine you have a recipe for making chocolate cake. A generative AI system could take that recipe and use its own algorithms to create a new recipe for making chocolate cakes. The ingredients and steps would still be the same, but the


In [5]:
chat_history = []

question = "What is Machine Learning?"

answer = generate_response(question, chat_history)

print("You:", question)
print("\nBot:", answer)

chat_history.extend([
    {"role": "user", "content": question},
    {"role": "assistant", "content": answer}
])

You: What is Machine Learning?

Bot: Machine learning is like teaching computers to learn and make decisions on their own, without being told what to do every single time. It's kind of like how you learned to ride a bike - you had someone teach you the first few steps, but eventually, you could balance and pedal all by yourself! In computer science, this process involves giving machines data and asking them to figure out patterns or rules based on those data points. The more data they receive, the better they get at making good decisions. This way, machines can become smarter over time, just like you got wiser with practice.


In [6]:
question = "Give me a real-life example of it."

answer = generate_response(question, chat_history)

print("You:", question)
print("\nBot:", answer)

chat_history.extend([
    {"role": "user", "content": question},
    {"role": "assistant", "content": answer}
])

You: Give me a real-life example of it.

Bot: A great example of machine learning is self-driving cars. Imagine if your car didn't have a human driver sitting behind the wheel. To drive safely, it needs to be able to detect obstacles, road conditions, and pedestrians. 

A company called Waymo uses machine learning to teach its cars to recognize these things through lots of training data. They show the cars pictures of different roads, people, and other objects. Then, when the car drives for the first time, it starts learning from that data. Over time, it becomes faster and more accurate at driving itself. 

For instance


In [9]:
def start_chat():

    history = []

    print("=" * 50)
    print("       OFFLINE GENAI CHATBOT")
    print("=" * 50)
    print("Ask any question.")
    print("Type /reset to clear history.")
    print("Type /exit to stop.")
    print("=" * 50)

    while True:

        question = input("\nYou: ").strip()

        if not question:
            continue

        if question.lower() == "/exit":
            print("Bot: Goodbye!")
            break

        if question.lower() == "/reset":
            history.clear()
            print("Bot: Conversation history cleared.")
            continue

        print("Bot: Thinking...")

        answer = generate_response(question, history)

        print("\nBot:", answer)

        history.extend([
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ])

        # Keep the history bounded
        if len(history) > 12:
            history = history[-12:]


# Run the chatbot
start_chat()

       OFFLINE GENAI CHATBOT
Ask any question.
Type /reset to clear history.
Type /exit to stop.



You:  As a student explain me what is SQL


Bot: Thinking...

Bot: SQL, which stands for Structured Query Language, is a programming language used to manage and manipulate data within databases. It allows users to create, edit, and query databases, as well as perform various operations on the data stored within them. 

Think of it like this: Imagine you have a big box of boxes with all sorts of items inside. SQL acts like a special key that unlocks the door to those boxes, allowing you to access the information inside and do things like counting how many books are in each shelf or knowing where to find specific items when you need them.

In



You:  Explain what is Agentic AI


Bot: Thinking...

Bot: Agentic AI refers to artificial intelligence systems that can act autonomously without human intervention or control. This means they can make decisions based on their own logic, rules, and algorithms, rather than being programmed by humans. The term "agentic" comes from the idea of agents - individuals who make choices and take actions independently, often driven by self-interest or preferences.

In the context of AI, agentic AI involves the ability of these systems to adapt to changing situations, learn from experience, and apply knowledge in real-time. They don't rely solely on pre-programmed instructions but



You:  /exit


Bot: Goodbye!
